In [2]:
import json
from azure.cognitiveservices.personalizer import PersonalizerClient
from azure.cognitiveservices.personalizer.models import RankRequest
from msrest.authentication import CognitiveServicesCredentials
from exe_functions import build_path
import os

from azure.cognitiveservices.personalizer import PersonalizerClient
from azure.cognitiveservices.personalizer.models import RankableAction, RewardRequest, RankRequest
from msrest.authentication import CognitiveServicesCredentials

import datetime, json, os, time, uuid, random

with open(build_path(os.path.abspath(os.curdir) + ("\\.keys"), "azure-personalizer-key.txt"), 'r') as f:
     personalizer_key = f.read().rstrip()
client = PersonalizerClient(
    "https://bwh-pharmacoepi-roybal-dev-e2-ehr-cog.cognitiveservices.azure.com/", 
    CognitiveServicesCredentials(personalizer_key)
)

key = personalizer_key
endpoint = client

# Instantiate a Personalizer client
#client = PersonalizerClient(endpoint, CognitiveServicesCredentials(key))

In [2]:
actions_and_features = {
    'pasta': {
        'brand_info': {
            'company':'pasta_inc'
        }, 
        'attributes': {
            'qty':1, 'cuisine':'italian',
            'price':12
        },
        'dietary_attributes': {
            'vegan': False,
            'low_carb': False,
            'high_protein': False,
            'vegetarian': False,
            'low_fat': True,
            'low_sodium': True
        }
    },
    'bbq': {
        'brand_info' : {
            'company': 'ambisco'
        },
        'attributes': {
            'qty': 2,
            'category': 'bbq',
            'price': 20
        }, 
        'dietary_attributes': {
            'vegan': False,
            'low_carb': True,
            'high_protein': True,
            'vegetarian': False,
            'low_fat': False,
            'low_sodium': False
        }
    },
    'bao': {
        'brand_info': {
            'company': 'bao_and_co'
        },
        'attributes': {
            'qty': 4,
            'category': 'chinese',
            'price': 8
        }, 
        'dietary_attributes': {
            'vegan': False,
            'low_carb': True,
            'high_protein': True,
            'vegetarian': False,
            'low_fat': True,
            'low_sodium': False
        }
    },
    'hummus': {
        'brand_info' : { 
            'company': 'garbanzo_inc'
        },
        'attributes' : {
            'qty': 1,
            'category': 'breakfast',
            'price': 5
        }, 
        'dietary_attributes': {
            'vegan': True, 
            'low_carb': False,
            'high_protein': True,
            'vegetarian': True,
            'low_fat': False, 
            'low_sodium': False
        }
    },
    'veg_platter': {
        'brand_info': {
            'company': 'farm_fresh'
        }, 
        'attributes': {
            'qty': 1,
            'category': 'produce', 
            'price': 7
        },
        'dietary_attributes': {
            'vegan': True,
            'low_carb': True,
            'high_protein': False,
            'vegetarian': True,
            'low_fat': True,
            'low_sodium': True
        }
    }
}

def get_actions():
    res = []
    for action_id, feat in actions_and_features.items():
        action = RankableAction(id=action_id, features=[feat])
        res.append(action)
    return res

user_profiles = {
    'nudgeid_011': {
            'demographics':{
                'sex_pcp': 'male', 
                'race_pcp_cat': 'black_or_african_american',
                'providertype_cat': 'physician',
                'specialty_cat': 'family_medicine',
                'yearsatatrius_int': 3
            },
            'practice_stats':{
                'panelsize_int': 1200,
                'prop_patient65plus': 0.3,
                'avg_patientage': 55,
                'avg_nb_patient_problems': 7,
                'avg_nb_appointments': 170,
                'avg_pct_encounters_closed_same_day': 0.9,
                'avg_pct_orders_contrib_other_providers': 0.25,
                'avg_doc_length_per_appt': 5239.91764705882,
                'avg_meds_per_appt_signed': 0,
                'avg_meds_per_appt_pended': 0,
                'avg_min_in_ehr_workday': 200,
                'avg_min_in_ehr_outisde_7a7p': 210,
                'avg_min_notes_appt': 5,
                'avg_min_inbasket_appt': 3,
                'avg_min_order_appt': 4,
                'avg_min_clinreview_appt': 11,
                'avg_min_unscheduled_days': 67,
                'avg_pct_orders_smartset': 0.1,
                'admin_fte': 0
            },
            'patients':{
                'nudgepatid_01055':{
                    'pat_age': 71,
                    'pat_sex': "male",
                    'pat_race': "white",
                    'pat_language': "english",
                    'encounter_weekday': "m",
                    'encounter_time': "am",
                    'hosp_last90days_yn': 1,
                    'er_visit_last90days_yn': 1,
                    'dementia_yn': 0,
                    'depression_yn': 0,
                    'anxiety_yn': 0,
                    'chronicpain_yn': 0,
                    'insomnia_yn': 0,
                    'samepcp_yn': 1,
                    'days_since_last_pcpvisit': 14,
                    'nb_pcp_visits_365days': 6,
                    'pcp_prescribed_highriskmed_yn': 1,
                    'nb_eligible_meds': 1,
                    'benzo_yn': 1,
                    'sedativehypnotic_yn': 0,
                    'anticholinergic_yn': 0,
                    'nb_pills_last180days': 160
                },
                'nudgepatid_12345':{
                    'pat_age': 89,
                    'pat_sex': " female",
                    'pat_race': " white",
                    'pat_language': " english",
                    'encounter_weekday': " tu",
                    'encounter_time': " am",
                    'hosp_last90days_yn':  0,
                    'er_visit_last90days_yn':  0,
                    'dementia_yn':  1,
                    'depression_yn':  0,
                    'anxiety_yn':  1,
                    'chronicpain_yn':  0,
                    'insomnia_yn':  0,
                    'samepcp_yn':  1,
                    'days_since_last_pcpvisit':  31,
                    'nb_pcp_visits_365days':  6,
                    'pcp_prescribed_highriskmed_yn':  1,
                    'nb_eligible_meds':  1,
                    'benzo_yn':  0,
                    'sedativehypnotic_yn':  1,
                    'anticholinergic_yn':  0,
                    'nb_pills_last180days':  160 
                },
                'nudgepatid_98765':{
                    'pat_age': 94,
                    'pat_sex': "female",
                    'pat_race': "white",
                    'pat_language': "english",
                    'encounter_weekday': "th",
                    'encounter_time': "am",
                    'hosp_last90days_yn': 0,
                    'er_visit_last90days_yn': 0,
                    'dementia_yn': 0,
                    'depression_yn': 1,
                    'anxiety_yn': 0,
                    'chronicpain_yn': 1,
                    'insomnia_yn': 0,
                    'samepcp_yn': 1,
                    'days_since_last_pcpvisit': 185,
                    'nb_pcp_visits_365days': 1,
                    'pcp_prescribed_highriskmed_yn': 0,
                    'nb_eligible_meds': 3,
                    'benzo_yn': 1,
                    'sedativehypnotic_yn': 1,
                    'anticholinergic_yn': 1,
                    'nb_pills_last180days': 130
                }
            },
    },
    'nudgeid_123': {
            'demographics':{
                'sex_pcp': 'male', 
                'race_pcp_cat': 'white',
                'providertype_cat': 'physician',
                'specialty_cat': 'internal_medicine',
                'yearsatatrius_int': 11
            },
            'practice_stats':{
                'panelsize_int': 2500,
                'prop_patient65plus': 0.6,
                'avg_patientage': 68,
                'avg_nb_patient_problems': 8,
                'avg_nb_appointments': 250,
                'avg_pct_encounters_closed_same_day': 0.7,
                'avg_pct_orders_contrib_other_providers': 0.15,
                'avg_doc_length_per_appt': 3924.228,
                'avg_meds_per_appt_signed': 1,
                'avg_meds_per_appt_pended': 0,
                'avg_min_in_ehr_workday': 190,
                'avg_min_in_ehr_outisde_7a7p': 160,
                'avg_min_notes_appt': 3,
                'avg_min_inbasket_appt': 2,
                'avg_min_order_appt': 8,
                'avg_min_clinreview_appt': 12,
                'avg_min_unscheduled_days': 25,
                'avg_pct_orders_smartset': 0.2,
                'admin_fte': 0.5
            }
    },
    'nudgeid_555': {
            'demographics':{
                'sex_pcp': 'female', 
                'race_pcp_cat': 'hispanic_or_latino',
                'providertype_cat': 'osteopath',
                'specialty_cat': 'family_medicine',
                'yearsatatrius_int': 7
            },
            'practice_stats':{
                'panelsize_int': 1850,
                'prop_patient65plus': 0.75,
                'avg_patientage': 72,
                'avg_nb_patient_problems': 5,
                'avg_nb_appointments': 162,
                'avg_pct_encounters_closed_same_day': 0.66,
                'avg_pct_orders_contrib_other_providers': 0.18,
                'avg_doc_length_per_appt': 7132.75925925926,
                'avg_meds_per_appt_signed': 0,
                'avg_meds_per_appt_pended': 1,
                'avg_min_in_ehr_workday': 170,
                'avg_min_in_ehr_outisde_7a7p': 177,
                'avg_min_notes_appt': 8,
                'avg_min_inbasket_appt': 1,
                'avg_min_order_appt': 5,
                'avg_min_clinreview_appt': 9,
                'avg_min_unscheduled_days': 45,
                'avg_pct_orders_smartset': 0.15,
                'admin_fte': 0.6
            }
    }
}

def get_context(user):
    location_context = {'location': random.choice(['west', 'east', 'midwest'])}
    time_of_day = {'time_of_day': random.choice(['morning', 'afternoon', 'evening'])}
    app_type = {'application_type': random.choice(['edge', 'safari', 'edge_mobile', 'mobile_app'])}
    res = [user_profiles[user], location_context, time_of_day, app_type]
    return res

def get_random_users(k = 5):
    return random.choices(list(user_profiles.keys()), k=k)


def get_reward_score(user, actionid, context):
    reward_score = 0.0
    action = actions_and_features[actionid]
    
    if user == 'Bill':
        if action['attributes']['price'] < 10 and (context[1]['location'] !=  "midwest"):
            reward_score = 1.0
            print("Bill likes to be economical when he's not in the midwest visiting his friend Warren. He bought", actionid, "because it was below a price of $10.")
        elif (action['dietary_attributes']['low_carb'] == True) and (context[1]['location'] ==  "midwest"):
            reward_score = 1.0
            print("Bill is visiting his friend Warren in the midwest. There he's willing to spend more on food as long as it's low carb, so Bill bought" + actionid + ".")
            
        elif (action['attributes']['price'] >= 10) and (context[1]['location'] != "midwest"):
            print("Bill didn't buy", actionid, "because the price was too high when not visting his friend Warren in the midwest.")
            
        elif (action['dietary_attributes']['low_carb'] == False) and (context[1]['location'] ==  "midwest"):
            print("Bill didn't buy", actionid, "because it's not low-carb, and he's in the midwest visitng his friend Warren.")
             
    elif user == 'Satya':
        if action['dietary_attributes']['low_sodium'] == True:
            reward_score = 1.0
            print("Satya is health conscious, so he bought", actionid,"since it's low in sodium.")
        else:
            print("Satya did not buy", actionid, "because it's not low sodium.")   
            
    elif user == 'Amy':
        if (action['dietary_attributes']['vegan'] == True) or (action['dietary_attributes']['vegetarian'] == True):
            reward_score = 1.0
            print("Amy likes to eat plant-based foods, so she bought", actionid, "because it's vegan or vegetarian friendly.")       
        else:
            print("Amy did not buy", actionid, "because it's not vegan or vegetarian.")
                
    return reward_score

In [3]:
def run_personalizer_cycle():
    actions = get_actions()
    user_list = get_random_users()
    for user in user_list:
        print("------------")
        print("User:", user, "\n")
        context = get_context(user)
        print("Context:", context, "\n")
        
        rank_request = RankRequest(actions=actions, context_features=context)
        response = client.rank(rank_request=rank_request)
        print("Rank API response:", response, "\n")
        
        eventid = response.event_id
        actionid = response.reward_action_id
        print("Personalizer recommended action", actionid, "and it was shown as the featured product.\n")
        
        reward_score = get_reward_score(user, actionid, context)
        client.events.reward(event_id=eventid, value=reward_score)     
        print("\nA reward score of", reward_score , "was sent to Personalizer.")
        print("------------\n")

continue_loop = True
while continue_loop:
    run_personalizer_cycle()
    
    br = input("Press Q to exit, or any other key to run another loop: ")
    if(br.lower()=='q'):
        continue_loop = False
# </snippet_2>

# <snippet_multi>
for i in range(0,10):
    run_personalizer_cycle()
# </snippet_multi>

------------
User: Bill 

Context: [{'dietary_preferences': 'low_carb', 'avg_order_price': '0-20', 'browser_type': 'edge'}, {'location': 'east'}, {'time_of_day': 'morning'}, {'application_type': 'mobile_app'}] 

Rank API response: {'additional_properties': {}, 'ranking': [<azure.cognitiveservices.personalizer.models.ranked_action_py3.RankedAction object at 0x000002CAEFCAB8C0>, <azure.cognitiveservices.personalizer.models.ranked_action_py3.RankedAction object at 0x000002CAEFD65090>, <azure.cognitiveservices.personalizer.models.ranked_action_py3.RankedAction object at 0x000002CAEFD64190>, <azure.cognitiveservices.personalizer.models.ranked_action_py3.RankedAction object at 0x000002CAEFD45480>, <azure.cognitiveservices.personalizer.models.ranked_action_py3.RankedAction object at 0x000002CAEFD455B0>], 'event_id': 'ae9930bbb0714cbdb2da95f7a1e0e228-fPyby', 'reward_action_id': 'veg_platter'} 

Personalizer recommended action veg_platter and it was shown as the featured product.

Bill likes to